# Notebook 03 — Model

Fits ICAR variants A-G, runs diagnostics (Moran I, VIF, DIC comparison),
generates risk rasters, optionally projects future deforestation, and runs GWR.
Requires outputs from `02_process.ipynb`.

In [ ]:
# papermill parameters — overridden at runtime
config_path = "configs/template.yaml"
runs_root = "runs"
run_dir = None   # if set, loads an existing run instead of creating a new one

In [ ]:
import pandas as pd
import forestatrisk as far
from palmoil_risk.io.run import create_run, load_run

if run_dir:
    ctx = load_run(run_dir)
else:
    ctx = create_run(config_path, runs_root=runs_root)
print(f"Run dir: {ctx.run_dir}")

In [ ]:
# Build sample dataset from aligned data/ directory
data = far.data.sample.sample(
    nsamp=10000,
    adapt=True,
    seed=1234,
    csize=ctx.config.csize,
    var_dir=str(ctx.data_dir),
    input_forest_raster=str(ctx.data_dir / "fcc23.tif"),
    output_file=str(ctx.output_dir / "sample.csv"),
    blk_rows=128,
)
data = pd.read_csv(ctx.output_dir / "sample.csv")
print(f"Sample size: {len(data)}")

In [ ]:
from palmoil_risk.model.icar import fit_icar

fitted = {}
for variant in ctx.config.model_variants:
    print(f"Fitting variant {variant}...")
    mod = fit_icar(ctx, variant, data.copy())
    fitted[variant] = {"model": mod}
    print(f"  variant {variant}: DIC={getattr(mod, 'DIC', 'N/A'):.1f}")
print("All variants fitted")

In [ ]:
from palmoil_risk.model.diagnostics import (
    compile_dic_table, check_vif, check_beta_stability
)

dic_df = compile_dic_table(fitted, ctx)
print("DIC comparison:")
print(dic_df.to_string(index=False))

best_variant = dic_df.iloc[0]["variant"]
print(f"\nBest variant by DIC: {best_variant}")

In [ ]:
# VIF on baseline covariates
baseline_terms = [
    "altitude", "slope",
    "dist_defor", "dist_edge", "dist_road", "dist_town", "dist_river",
]
vifs = check_vif(data, baseline_terms)
print("VIF:", {k: round(v, 2) for k, v in vifs.items()})

In [ ]:
# Moran's I on ICAR residuals for best variant
import numpy as np
from palmoil_risk.model.diagnostics import compute_moran

best_mod = fitted[best_variant]["model"]
if hasattr(best_mod, "rho"):
    residuals = np.array(best_mod.rho)
    coords = data[["x", "y"]].values if "x" in data.columns else None
    if coords is not None:
        moran = compute_moran(residuals, coords, ctx)
        print(f"Moran's I: {moran['moran_i']:.4f} ({moran['interpretation']})")
else:
    print("Moran's I skipped: no rho attribute on model")

In [ ]:
from palmoil_risk.model.predict import predict_risk, project_future

risk_paths = {}
for variant, info in fitted.items():
    pkl_path = ctx.output_dir / "models" / f"model_{variant}" / f"mod_{variant}.pkl"
    risk_path = predict_risk(ctx, pkl_path, variant)
    risk_paths[variant] = risk_path
    print(f"Risk map written: {risk_path}")

    if ctx.config.project_future:
        fut = project_future(ctx, risk_path, variant)
        if fut:
            print(f"  Future forest: {fut}")

In [ ]:
from palmoil_risk.model.gwr import run_gwr

if ctx.config.run_gwr:
    gwr_result = run_gwr(
        ctx, data,
        y_col="fcc23",
        x_cols=["altitude", "slope", "dist_defor", "dist_edge"],
        coords_cols=("x", "y"),
    )
    if gwr_result:
        print(f"GWR: bandwidth={gwr_result['bandwidth']:.0f}m  R2={gwr_result['r2']:.4f}")
else:
    print("GWR skipped (run_gwr=False)")